# Matching Embeddings (TF two-tower + FAISS)

# Matching Embeddings — TF two-tower + FAISS index (Food.com interactions)

Recsys-style contrastive two-tower on **Food.com user↔recipe interactions**
(`RAW_interactions.csv`, ~1.1M ratings). Builds the FAISS index consumed by
`app/embedding_engine.py` `/api/v1/embeddings/match`. While the index is stale
the serving fallback path stays the offline route.

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'faiss'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}
from tf_utils import set_memory_growth
set_memory_growth()


In [ ]:
from buddy_data import food_com_interactions
inter = food_com_interactions()
print('interactions:', inter.shape)
inter.head(3)

In [ ]:
import pandas as pd
users = pd.Series(inter['user_id'].astype('category').cat.codes.values)
items = pd.Series(inter['recipe_id'].astype('category').cat.codes.values)
n_users, n_items = users.nunique(), items.nunique()
print('users:', n_users, 'items:', n_items)

In [ ]:
import tensorflow as tf
from tf_utils import build_two_tower
m = build_two_tower(embed_dim=128, n_users=n_users, n_items=n_items)
m.summary()

In [ ]:
# Train with a small margin (rating >= 3 => positive pair).
m.fit([users, items], inter['rating'].astype('float32'), epochs=10, validation_split=0.05)

In [ ]:
import numpy as np, faiss
item_emb = m.layers[2].get_weights()[0]          # item embedding matrix
index = faiss.IndexFlatIP(item_emb.shape[1])
index.add(item_emb.astype('float32'))
faiss.write_index(index, '../models/matching_items.faiss')
print('FAISS index items:', index.ntotal)

In [ ]:
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log
onnx = export_keras_onnx(m, Path('../models'), 'matching_embeddings', '1.0.0')
mlflow_log({'name':'matching_embeddings','version':'1.0.0','artifact_path':str(onnx),
            'framework':'tensorflow','metrics':{}})